In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/25 23:41:54 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/25 23:41:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2a0d19be-5c0b-488f-a0a2-c80c030a016b;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 69ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#

In [2]:
from datetime import date

from pyspark.sql import functions as F

from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS, 
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient


def display_df(df):
    display(df.toPandas())

In [5]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
    run_tracker=run_tracker,
)

In [6]:
pipeline_result = pipeline.execute()

print(f"Pipeline run complete: {pipeline_result}")

business_dt = pipeline._config.business_dt
batch_id = pipeline._config.batch_id

Pipeline run complete: PipelineResult(identity=RunIdentity(workflow_run_id=UUID('c5b85e4d-c7c1-42aa-9eef-c037553c5f73'), run_id=UUID('eb961411-0eae-482f-a0ac-0cd97de34d67'), parent_run_id=None), status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, zones=(ZoneResult(identity=RunIdentity(workflow_run_id=UUID('c5b85e4d-c7c1-42aa-9eef-c037553c5f73'), run_id=UUID('84741cd3-ec50-44c6-af06-0af04f76c8d2'), parent_run_id=UUID('eb961411-0eae-482f-a0ac-0cd97de34d67')), zone='STAGING', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=42), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('c5b85e4d-c7c1-42aa-9eef-c037553c5f73'), run_id=UUID('8594c608-1206-4dce-b000-4f1c2e73cdc0'), parent_run_id=UUID('eb961411-0eae-482f-a0ac-0cd97de34d67')), zone='ENRICHMENT', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=7), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('c5b85e4d-c7c1-42aa-9eef-c037553c5f73'), run_id=UUID('2572a8e0-95cf-475c-ac3c-d65dd3382e2c'), parent_run_id=UUID('eb961411-0eae-48

In [7]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,SRC_CLIENT_ID,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,90000.000000000000,USD
1,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,15000.000000000000,USD
2,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,-5000.000000000000,USD
3,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,100000.000000000000,USD
4,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_BACK_VALUED_ADJUSTMENT,USD,0E-12,USD
5,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_ADJUSTED_BALANCE,USD,100000.000000000000,USD
6,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,15000.000000000000,USD
7,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,5000.000000000000,USD
8,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
9,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,20000.000000000000,USD


In [8]:
stg_df = repository.read_staging(business_dt, batch_id)

display_df(stg_df)

,BATCH_ID,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,100000.000000000000,1.000000000000,100000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
1,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
2,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,-5000.000000000000,1.000000000000,-5000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
3,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,15000.000000000000,1.000000000000,15000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
4,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,100000.000000000000,1.000000000000,100000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
5,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,USD,PREVIOUS_DAY_BALANCE,REPORTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
6,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,20000.000000000000,1.000000000000,20000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
7,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
8,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2
9,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,5000.000000000000,1.000000000000,5000.000000000000,DEBIT,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,84741cd3-ec50-44c6-af06-0af04f76c8d2


In [9]:
enr_df = repository.read_enrichment(business_dt, batch_id)

display_df(enr_df)

,BATCH_ID,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0
1,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0
2,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,USM,TRD,2000,LIABILITY,...,130000,210000,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0
3,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,USM,FIN,4000,REVENUE,...,410000,410000,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0
4,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,USM,FIN,3000,EQUITY,...,310000,310000,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0
5,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,CAM,TRD,1000,ASSET,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0
6,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,CAM,TRD,2100,LIABILITY,...,140000,230000,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,8594c608-1206-4dce-b000-4f1c2e73cdc0


In [10]:
rpt_df = repository.read_reporting(business_dt, batch_id)

display_df(rpt_df)

,BATCH_ID,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
1,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
2,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
3,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
4,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
5,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
6,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
7,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
8,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c
9,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,...,,,,,,,,,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,2572a8e0-95cf-475c-ac3c-d65dd3382e2c


In [11]:
pst_df = repository.read_posting(business_dt, batch_id)

display_df(pst_df)

,BATCH_ID,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,POSTING_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,...,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,PST-260331-260331-2-7,USM,TRD,1000,...,0E-12,100000.000000000000,90000.000000000000,15000.000000000000,-5000.000000000000,100000.000000000000,0E-12,100000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f
1,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,PST-260331-260331-2-5,USM,FIN,1200,...,0E-12,20000.000000000000,15000.000000000000,5000.000000000000,0E-12,20000.000000000000,0E-12,20000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f
2,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,PST-260331-260331-2-1,USM,TRD,2000,...,0E-12,-70000.000000000000,-65000.000000000000,0E-12,-5000.000000000000,-70000.000000000000,0E-12,-70000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f
3,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,PST-260331-260331-2-6,USM,FIN,4000,...,0E-12,-30000.000000000000,-25000.000000000000,0E-12,-5000.000000000000,-30000.000000000000,0E-12,-30000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f
4,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,PST-260331-260331-2-4,USM,FIN,3000,...,0E-12,-20000.000000000000,-15000.000000000000,0E-12,-5000.000000000000,-20000.000000000000,0E-12,-20000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f
5,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,PST-260331-260331-2-3,CAM,TRD,1000,...,1000.000000000000,51000.000000000000,40000.000000000000,12000.000000000000,-2000.000000000000,50000.000000000000,1000.000000000000,51000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f
6,2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,PST-260331-260331-2-2,CAM,TRD,2100,...,-1000.000000000000,-51000.000000000000,-40000.000000000000,2000.000000000000,-12000.000000000000,-50000.000000000000,-1000.000000000000,-51000.000000000000,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,6337bad8-9d86-45dd-8162-49e8741d384f


In [12]:
int_df = repository.read_interface(business_dt, batch_id)

display_df(int_df)

,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,...,SRC_RECORD_ID,BATCH_ID,SRC_APP_CD,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-USM-260331,1,1000,1100,NYC,101000,1000,...,rec-1,2,NFM,USD,100000.000000000000,USD,100000.000000000000,1.000000000000,2026-03-31,2026-03-31
1,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-USM-260331,2,1000,1200,NYC,120000,1200,...,rec-2,2,NFM,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
2,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-USM-260331,3,1000,1100,NYC,210000,2000,...,rec-3,2,NFM,USD,-70000.000000000000,USD,-70000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-USM-260331,4,1000,1200,NYC,410000,4000,...,rec-4,2,NFM,USD,-30000.000000000000,USD,-30000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-USM-260331,5,1000,1200,NYC,310000,3000,...,rec-5,2,NFM,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-CAM-260331,1,2000,2100,TOR,101000,1000,...,rec-6,2,NFM,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
6,c5b85e4d-c7c1-42aa-9eef-c037553c5f73,cf24952a-1127-4b6e-817f-3379714f3d9e,TRIAL_BALANCE,NFTB-2-CAM-260331,2,2000,2100,TOR,230000,2100,...,rec-7,2,NFM,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31


In [11]:
# spark.stop()

In [17]:
def to_csv(df, file_name):
    (
        df.write
        .mode('overwrite')
        .option('header', True)
        .csv(f'data/sample/{file_name}.csv')
    )

In [23]:
to_csv(int_df, 'interface')